<a href="https://colab.research.google.com/github/fkhaaan/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Menu Detector!")

Menu Detector!


In [18]:
#=================
# Import Libraries
#=================
from google.colab import drive
import torchvision.transforms as transforms
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import os
import numpy as np

In [19]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("Dataset_path:", DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    "baklava": "baklava",
    "hot_dog": "hot_dog",
    "cheesecake": "dessert", # label grouping | class consolidation
    "pilaf": "pilaf",
    "pizza": "pizza",
    "chocolate_cake": "dessert" # label grouping | class consolidation
}

CLASSES = ['baklava', 'hot_dog', 'dessert', 'pilaf', 'pizza']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]
                         )
])



Dataset_path: /content/drive/MyDrive/food101_dataset


In [26]:
# ----------------------
# Custom Dataset Class
# ----------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('image_path', img_path)

        label = self.labels[idx]
        print('label', label)

        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [27]:
# ----------------------
# Gather and Split Data
# ----------------------

all_images = []

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)

    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))

img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/baklava
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/pilaf
class_path: /content/drive/MyDrive/food101_dataset/pizza
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
all_images: [('/content/drive/MyDrive/food101_dataset/chocolate_cake/3160967.jpg', 2), ('/content/drive/MyDrive/food101_dataset/baklava/3691584.jpg', 0), ('/content/drive/MyDrive/food101_dataset/baklava/3841998.jpg', 0), ('/content/drive/MyDrive/food101_dataset/hot_dog/1756794.jpg', 1), ('/content/drive/MyDrive/food101_dataset/hot_dog/930190.jpg', 1), ('/content/drive/MyDrive/food101_dataset/chocolate_cake/1404915.jpg', 2), ('/content/drive/MyDrive/food101_dataset/hot_dog/327851.jpg', 1), ('/content/drive/MyDrive/food101_dataset/hot_dog/935952.jpg', 1), ('/content/drive/MyDrive/food101_dataset/hot_dog/784362.jpg', 1), ('/content/dri